# Prosecutor Generation Notebook (Gemini + Hybrid Retrieval)
This notebook validates GEMINI_API_KEY, runs live prosecution generation, and regression-tests multiple criminal fact patterns.

In [1]:
from pathlib import Path
import sys

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        if (root / "backend" / "retrieval").exists():
            return root
    raise RuntimeError(
        "Project root not found from current working directory. "
        "Open this notebook from the Court_RAG_project folder."
    )

ROOT = resolve_project_root()
BACKEND_DIR = ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("Using project root:", ROOT)

Using project root: P:\6thSem\GENAI\project\code\Court_RAG_project


In [2]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

backend_env = ROOT / "backend" / ".env"
if backend_env.exists():
    load_dotenv(dotenv_path=str(backend_env), override=False)
    print("Loaded .env from:", backend_env)
else:
    dotenv_path = find_dotenv(usecwd=True)
    if dotenv_path:
        load_dotenv(dotenv_path=dotenv_path, override=False)
        print("Loaded .env from:", dotenv_path)
    else:
        load_dotenv(override=False)
        print("No .env located by backend path or find_dotenv(); tried default load_dotenv().")

gemini_key = (os.getenv("GEMINI_API_KEY") or "").strip()
if not gemini_key:
    print("ERROR: GEMINI_API_KEY not found in environment/.env")
else:
    print("GEMINI_API_KEY loaded from environment/.env")

Loaded .env from: P:\6thSem\GENAI\project\code\Court_RAG_project\backend\.env
GEMINI_API_KEY loaded from environment/.env


In [3]:
import re
import time
from importlib import reload
import prosecution_pipeline as pp

reload(pp)
prosecution_arguments = pp.prosecution_arguments
defence_arguments = pp.defence_arguments
judge_arguments = pp.judge_arguments

def sentence_count(text: str) -> int:
    return len([s for s in re.split(r"(?<=[.!?])\s+", (text or "").strip()) if s.strip()])

def run_case(user_case: str) -> dict:
    print("=" * 90)
    print("INPUT:", user_case)
    total_start = time.perf_counter()

    try:
        prosecution_start = time.perf_counter()
        prosecution_result = prosecution_arguments(user_case)
        prosecution_elapsed = time.perf_counter() - prosecution_start

        print("\n[PROSECUTOR OUTPUT]")
        print(f"Generated in {prosecution_elapsed:.2f}s")
        print(prosecution_result)

        defence_start = time.perf_counter()
        defence_result = defence_arguments(prosecutor_output=prosecution_result, user_case=user_case)
        defence_elapsed = time.perf_counter() - defence_start

        print("\n[DEFENCE OUTPUT]")
        print(f"Generated in {defence_elapsed:.2f}s")
        print(defence_result)

        judge_start = time.perf_counter()
        judge_result = judge_arguments(
            prosecutor_output=prosecution_result,
            defence_output=defence_result,
            user_case=user_case,
        )
        judge_elapsed = time.perf_counter() - judge_start

        print("\n[JUDGE OUTPUT]")
        print(f"Generated in {judge_elapsed:.2f}s")
        print(judge_result)

    except Exception as exc:
        elapsed = time.perf_counter() - total_start
        print(f"FAILED after {elapsed:.2f}s: {exc}")
        raise

    total_elapsed = time.perf_counter() - total_start
    print(f"\nCompleted full run in {total_elapsed:.2f}s")

    prosecution_checks = {
        "single_paragraph": "\n\n" not in prosecution_result.strip(),
        "has_ipc": bool(re.search(r"\bIPC\s*\d{1,3}[A-Z]?\b", prosecution_result, flags=re.IGNORECASE)),
        "sentence_count_ok": 8 <= sentence_count(prosecution_result) <= 18,
    }
    defence_checks = {
        "single_paragraph": "\n\n" not in defence_result.strip(),
        "has_ipc": bool(re.search(r"\bIPC\s*\d{1,3}[A-Z]?\b", defence_result, flags=re.IGNORECASE)),
        "sentence_count_ok": 8 <= sentence_count(defence_result) <= 18,
    }
    judge_checks = {
        "single_paragraph": "\n\n" not in judge_result.strip(),
        "has_ipc": bool(re.search(r"\bIPC\s*\d{1,3}[A-Z]?\b", judge_result, flags=re.IGNORECASE)),
        "has_specific_sentence": bool(re.search(r"\b(death penalty|imprisonment for life|life imprisonment|imprisonment for\s+\d+\s+years?|rigorous imprisonment for\s+\d+\s+years?)\b", judge_result, flags=re.IGNORECASE)),
        "has_compensation_amount": bool(re.search(r"(₹\s*\d[\d,]*|\b(?:INR|Rs\.?|Rupees)\s*\d[\d,]*)", judge_result, flags=re.IGNORECASE)),
        "sentence_count_ok": 8 <= sentence_count(judge_result) <= 18,
    }
    print("Prosecution checks:", prosecution_checks)
    print("Defence checks:", defence_checks)
    print("Judge checks:", judge_checks)

    return {
        "prosecution": prosecution_result,
        "defence": defence_result,
        "judge": judge_result,
    }

print("Helper ready: run_case(user_case) -> {'prosecution': ..., 'defence': ..., 'judge': ...}")

c:\Anaconda\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(



Helper ready: run_case(user_case) -> {'prosecution': ..., 'defence': ..., 'judge': ...}


P:\6thSem\GENAI\project\code\Court_RAG_project\backend\utils\llm.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
run_case("A man stabbed another person with a knife causing death.")

INPUT: A man stabbed another person with a knife causing death.
[Stage 1] Using device: cpu
[Stage 1] Loading base model: nlpaueb/legal-bert-base-uncased
[Stage 1] Could not load nlpaueb/legal-bert-base-uncased: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
[Stage 1] Loading base model: distilbert-base-uncased
[Stage 1] Could not load distilbert-base-uncased: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
[precedent_retriever] Loading embedding model: all-MiniLM-L6-v2 (CPU only)...
[precedent_retriever] Model loaded.
[statute_retriever] Loading embedding model: all

{'prosecution': "It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as A man stabbed another person with a knife causing death., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 304 (Punishment for culpable homicide not amounting to murder), IPC 299 (Culpable homicide), IPC 301 (Culpable homicide by causing death of person other than person whose death was intended), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from The State Of Andhra Pradesh vs Kunchala Sasi Krishna on 31 (convicted; IPC 354D, IPC 302, IPC 3, IPC 366), Harjeet Singh @ Popy And Sudhadhar @ Sofi ... vs State on 20 (convicted; IPC 302, IPC 34, IPC 3

In [5]:
run_case("During a heated street quarrel, the accused was slapped first, then pushed the victim, who fell, hit his head on a stone, and later died in hospital. There was no prior enmity, no weapon brought to scene, and eyewitnesses confirm the incident lasted under a minute.")

INPUT: During a heated street quarrel, the accused was slapped first, then pushed the victim, who fell, hit his head on a stone, and later died in hospital. There was no prior enmity, no weapon brought to scene, and eyewitnesses confirm the incident lasted under a minute.

[PROSECUTOR OUTPUT]
Generated in 0.77s
It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as During a heated street quarrel, the accused was slapped first, then pushed the victim, who fell, hit his head on a stone, and later died in hospital. There was no prior enmity, no weapon brought to scene, and eyewitnesses confirm the incident lasted under a minute., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 321 (Voluntarily causing hurt), IPC 334 (Voluntarily causing hurt on provocation), IPC 504 (Intenti

{'prosecution': "It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as During a heated street quarrel, the accused was slapped first, then pushed the victim, who fell, hit his head on a stone, and later died in hospital. There was no prior enmity, no weapon brought to scene, and eyewitnesses confirm the incident lasted under a minute., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 321 (Voluntarily causing hurt), IPC 334 (Voluntarily causing hurt on provocation), IPC 504 (Intentional insult with intent to provoke breach of the peace), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Gurmukh Singh vs State Of Har

In [7]:
run_case("Prosecution relies on one eyewitness identifying the accused at night from far distance, but tower location and toll records place accused in another district. CCTV timestamps conflict with eyewitness timeline.")

INPUT: Prosecution relies on one eyewitness identifying the accused at night from far distance, but tower location and toll records place accused in another district. CCTV timestamps conflict with eyewitness timeline.

[PROSECUTOR OUTPUT]
Generated in 0.97s
It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as Prosecution relies on one eyewitness identifying the accused at night from far distance, but tower location and toll records place accused in another district. CCTV timestamps conflict with eyewitness timeline., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 354C (Voyeurism), IPC 192 (Fabricating false evidence), IPC 193 (Punishment for false evidence), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant int

{'prosecution': "It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as Prosecution relies on one eyewitness identifying the accused at night from far distance, but tower location and toll records place accused in another district. CCTV timestamps conflict with eyewitness timeline., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 354C (Voyeurism), IPC 192 (Fabricating false evidence), IPC 193 (Punishment for false evidence), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Potnuru Appala Naidu, Vizianagaram ... vs P.P., Hyd on 6 (convicted; IPC 417, IPC 376, IPC 90, IPC 3), State vs . Akbar Malik on 29 April, 2017

In [6]:
run_case("At a wedding brawl, the accused was heavily intoxicated, picked up a bottle fragment, and caused fatal injury during chaotic group fighting. Witnesses disagree on who started the fight; no prior planning is shown.")

INPUT: At a wedding brawl, the accused was heavily intoxicated, picked up a bottle fragment, and caused fatal injury during chaotic group fighting. Witnesses disagree on who started the fight; no prior planning is shown.

[PROSECUTOR OUTPUT]
Generated in 0.91s
It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as At a wedding brawl, the accused was heavily intoxicated, picked up a bottle fragment, and caused fatal injury during chaotic group fighting. Witnesses disagree on who started the fight; no prior planning is shown., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 86 (Offence requiring a particular intent or knowledge committed by one who is intoxicated), IPC 85 (Act of a person incapable of judgment by reason of intoxication caused against his will), IPC 44 (Inju

{'prosecution': "It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as At a wedding brawl, the accused was heavily intoxicated, picked up a bottle fragment, and caused fatal injury during chaotic group fighting. Witnesses disagree on who started the fight; no prior planning is shown., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 86 (Offence requiring a particular intent or knowledge committed by one who is intoxicated), IPC 85 (Act of a person incapable of judgment by reason of intoxication caused against his will), IPC 44 (Injury), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Udai Singh vs State on 2 Dece

In [7]:
run_case("The accused fired at the victim during a dispute and the victim died on the spot.")

INPUT: The accused fired at the victim during a dispute and the victim died on the spot.

[PROSECUTOR OUTPUT]
Generated in 0.79s
It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused fired at the victim during a dispute and the victim died on the spot., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 302 (Punishment for murder), IPC 300 (Murder), IPC 211 (False charge of offence made with intent to injure), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Sayed Ahmed Ali Kari Alias Munna And Etc vs State Of (convicted; IPC 302, IPC 34, IPC 326, IPC 387), Satyavir Singh Rathi vs State Tr.C.B.I on 2 May

{'prosecution': "It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused fired at the victim during a dispute and the victim died on the spot., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 302 (Punishment for murder), IPC 300 (Murder), IPC 211 (False charge of offence made with intent to injure), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Sayed Ahmed Ali Kari Alias Munna And Etc vs State Of (convicted; IPC 302, IPC 34, IPC 326, IPC 387), Satyavir Singh Rathi vs State Tr.C.B.I on 2 May, 2011 (convicted; IPC 201, IPC 34, IPC 203, IPC 186), Amit vs State Nct Of Delhi on 10 November, 2014 (convicte

In [5]:
run_case("The accused forged property documents and cheated an elderly victim of savings.")

INPUT: The accused forged property documents and cheated an elderly victim of savings.

[PROSECUTOR OUTPUT]
Generated in 31.13s
It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused forged property documents and cheated an elderly victim of savings., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 471 (Using as genuine a forged document), IPC 467 (Forgery of valuable security, will, etc.), IPC 470 (Forged document), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Sh. Suneel Galgotia And Another vs State Of U.P. Thru Secy. (sentenced; IPC 420, IPC 467, IPC 468, IPC 471), M. Sivaram And Ors. vs State O

{'prosecution': "It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused forged property documents and cheated an elderly victim of savings., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 471 (Using as genuine a forged document), IPC 467 (Forgery of valuable security, will, etc.), IPC 470 (Forged document), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Sh. Suneel Galgotia And Another vs State Of U.P. Thru Secy. (sentenced; IPC 420, IPC 467, IPC 468, IPC 471), M. Sivaram And Ors. vs State Of A.P. And Anr. on 22 August, (sentenced; IPC 418, IPC 420, IPC 425, IPC 427), Kalavati Devi @ Kalavati vs Stat

In [9]:
run_case("The accused assaulted the victim with an iron rod causing grievous injuries but not death.")

INPUT: The accused assaulted the victim with an iron rod causing grievous injuries but not death.
Completed in 33.11s
It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused assaulted the victim with an iron rod causing grievous injuries but not death., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 325 (Punishment for voluntarily causing grievous hurt), IPC 87 (Act not intended and not known to be likely to cause death or grievous hurt, done by consent), IPC 326 (Voluntarily causing grievous hurt by dangerous weapons or means), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Ram Singh & Ors. vs The S

"It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused assaulted the victim with an iron rod causing grievous injuries but not death., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 325 (Punishment for voluntarily causing grievous hurt), IPC 87 (Act not intended and not known to be likely to cause death or grievous hurt, done by consent), IPC 326 (Voluntarily causing grievous hurt by dangerous weapons or means), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Ram Singh & Ors. vs The State Of M.P on 5 January, 2012 (convicted; IPC 326, IPC 34, IPC 325, IPC 324), Joy vs State Of Kerala (sentenced; IPC

In [8]:
run_case("The accused entered the victim's house at night, attacked with a weapon, and caused fatal injuries.")

INPUT: The accused entered the victim's house at night, attacked with a weapon, and caused fatal injuries.

[PROSECUTOR OUTPUT]
Generated in 0.49s
It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused entered the victim's house at night, attacked with a weapon, and caused fatal injuries., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 460 (All persons jointly concerned in lurking house-trespass or house-breaking by night punishable where death or grievous hurt caused), IPC 457 (Lurking house-trespass or house-breaking by night in order to commit offence punishable with imprisonment), IPC 458 (Lurking house-trespass or house-breaking by night after preparation for hurt, assault, or wrongful restraint), and these are the principal penal provisions that precise

{'prosecution': "It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused entered the victim's house at night, attacked with a weapon, and caused fatal injuries., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 460 (All persons jointly concerned in lurking house-trespass or house-breaking by night punishable where death or grievous hurt caused), IPC 457 (Lurking house-trespass or house-breaking by night in order to commit offence punishable with imprisonment), IPC 458 (Lurking house-trespass or house-breaking by night after preparation for hurt, assault, or wrongful restraint), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial

In [8]:
run_case("The younger brother was jealous of the elder brother's success, so he poisoned the elder brother's food, leading to his death.")

INPUT: The younger brother was jealous of the elder brother's success, so he poisoned the elder brother's food, leading to his death.

[PROSECUTOR OUTPUT]
Generated in 0.43s
It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The younger brother was jealous of the elder brother's success, so he poisoned the elder brother's food, leading to his death., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 304A (Causing death by negligence), IPC 301 (Culpable homicide by causing death of person other than person whose death was intended), IPC 284 (Negligent conduct with respect to poisonous substance), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparabl

{'prosecution': "It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The younger brother was jealous of the elder brother's success, so he poisoned the elder brother's food, leading to his death., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 304A (Causing death by negligence), IPC 301 (Culpable homicide by causing death of person other than person whose death was intended), IPC 284 (Negligent conduct with respect to poisonous substance), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequence in law. Comparable judicial guidance emerges from Madan Gopal Kakkad vs Naval Dubey And Anr on 29 April, 1992 (convicted; IPC 376, IPC 354, IPC 511, IPC 375), The State Of Ka

In [9]:
run_case("The accused entered the victim's house at night after hearing cries for help. The victim attacked first with a weapon, and during the struggle, the accused acted in self-defense, resulting in the victim's death.")

INPUT: The accused entered the victim's house at night after hearing cries for help. The victim attacked first with a weapon, and during the struggle, the accused acted in self-defense, resulting in the victim's death.

[PROSECUTOR OUTPUT]
Generated in 0.51s
It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused entered the victim's house at night after hearing cries for help. The victim attacked first with a weapon, and during the struggle, the accused acted in self-defense, resulting in the victim's death., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 458 (Lurking house-trespass or house-breaking by night after preparation for hurt, assault, or wrongful restraint), IPC 460 (All persons jointly concerned in lurking house-trespass or house-breaking by night

{'prosecution': "It is respectfully submitted that the victim in the present case suffered grave harm in the incident narrated as The accused entered the victim's house at night after hearing cries for help. The victim attacked first with a weapon, and during the struggle, the accused acted in self-defense, resulting in the victim's death., and the emotional and physical devastation caused to the victim and family calls for firm judicial response rooted in both compassion and accountability. The gravamen of the offence squarely falls within IPC 458 (Lurking house-trespass or house-breaking by night after preparation for hurt, assault, or wrongful restraint), IPC 460 (All persons jointly concerned in lurking house-trespass or house-breaking by night punishable where death or grievous hurt caused), IPC 446 (House-breaking by night), and these are the principal penal provisions that precisely characterize the unlawful conduct, the attendant intent or knowledge, and the resulting consequen